# PCA & Manifold Learning — Applied on Real Dataset
**Digits Dataset: Dimensionality Reduction + Classification**

---
## Table of Contents
1. Dataset Overview — Digits Dataset
2. EDA — High-Dimensional Data Visualization
3. PCA — Dimensionality Reduction
4. Reconstruction — PCA se Original Image Recover karna
5. PCA + Classification — Pipeline
6. t-SNE — Cluster Visualization
7. PCA vs t-SNE Comparison
8. Capstone Challenge + Solutions
---

## Section 1: Dataset Overview

### Keywords:
- **Digits Dataset:** 8×8 pixel handwritten digit images (0-9)
- **64 features:** Each pixel is a feature (8×8 = 64 pixels)
- **Grayscale:** 0 = black, 16 = white pixel values
- **High Dimensional:** 64 features — hard to visualize
- **Compression Goal:** 64D → 2D/3D while keeping most info

In [ ]:
from sklearn.datasets import load_digits
import numpy as np
import matplotlib.pyplot as plt

digits = load_digits()
X = digits.data     # shape: (1797, 64)
y = digits.target   # shape: (1797,) — values 0-9

print(f"Dataset shape: {X.shape}")
print(f"Pixel range: {X.min()} to {X.max()}")
print(f"Classes: {np.unique(y)} (digits 0-9)")
print(f"\nClass distribution:")
for digit in range(10):
    count = (y == digit).sum()
    print(f"  Digit {digit}: {count} samples")

# Show some sample images
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    idx = np.where(y == digit)[0][0]
    axes[0, digit].imshow(digits.images[idx], cmap='gray_r')
    axes[0, digit].set_title(f'Digit: {digit}', fontsize=9)
    axes[0, digit].axis('off')
    # Second example
    idx2 = np.where(y == digit)[0][1]
    axes[1, digit].imshow(digits.images[idx2], cmap='gray_r')
    axes[1, digit].axis('off')

plt.suptitle('Digits Dataset — Sample Images (8×8 pixels each)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 2: EDA — High-Dimensional Data

In [ ]:
# Average face of each digit
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for digit in range(10):
    avg_image = X[y == digit].mean(axis=0).reshape(8, 8)
    axes[digit].imshow(avg_image, cmap='gray_r')
    axes[digit].set_title(f'{digit}', fontsize=11, fontweight='bold')
    axes[digit].axis('off')

plt.suptitle('Average Image per Digit Class', fontsize=11)
plt.tight_layout()
plt.show()

# Pixel importance heatmap
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in zip(range(10), axes.flatten()):
    std_image = X[y == digit].std(axis=0).reshape(8, 8)
    im = ax.imshow(std_image, cmap='hot')
    ax.set_title(f'Digit {digit}\nVariance')
    ax.axis('off')

plt.suptitle('Pixel Variance per Digit (Hot = High Variance)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 3: PCA — Dimensionality Reduction

### Keywords:
- **Eigenface:** PCA component for face/image data — each PC is a 'proto-image'
- **Explained Variance Ratio:** PC kitna % information rakhta hai
- **Cumulative Variance:** Total info kept with k PCs

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Scale first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA with all components
pca_full = PCA()
pca_full.fit(X_scaled)

# Scree plot
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, 65), pca_full.explained_variance_ratio_ * 100, 'bo-', markersize=4)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot')
axes[0].set_xlim(0, 30)
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, 65), cumvar, 'rs-', markersize=4)
for threshold in [80, 90, 95]:
    n_comp = np.argmax(cumvar >= threshold) + 1
    axes[1].axhline(threshold, linestyle='--', alpha=0.7, label=f'{threshold}%: {n_comp} PCs')
axes[1].set_xlabel('Number of PCs')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

for threshold in [80, 90, 95]:
    n = np.argmax(cumvar >= threshold) + 1
    print(f"{threshold}% variance needs {n} components (from 64) — {n/64*100:.0f}% of original features")

In [ ]:
# Visualize Principal Components (Eigendigits)
pca_20 = PCA(n_components=20)
pca_20.fit(X_scaled)

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i, ax in enumerate(axes.flatten()):
    pc_image = pca_20.components_[i].reshape(8, 8)
    ax.imshow(pc_image, cmap='RdBu_r')
    ax.set_title(f'PC{i+1}\n{pca_20.explained_variance_ratio_[i]*100:.1f}%', fontsize=7)
    ax.axis('off')

plt.suptitle('Principal Components (Eigendigits) — Each captures a pattern', fontsize=11)
plt.tight_layout()
plt.show()

print("Observation:")
print("- PC1 captures global brightness pattern")
print("- PC2 captures horizontal vs vertical strokes")
print("- Higher PCs capture finer details")

## Section 4: Reconstruction — PCA se Image Recover karna

### Keywords:
- **Reconstruction:** Compressed data se original data wapas banana
- **`inverse_transform`:** PCA space → original feature space
- **Compression Ratio:** kitne PCs use kiye / original features
- **Reconstruction Error:** Original vs Reconstructed ka difference

In [ ]:
# Show reconstruction quality at different n_components
n_comp_values = [1, 5, 10, 20, 40, 64]
sample_idx = 10  # sample digit to reconstruct

fig, axes = plt.subplots(1, len(n_comp_values) + 1, figsize=(14, 2.5))

# Original
axes[0].imshow(X[sample_idx].reshape(8, 8), cmap='gray_r')
axes[0].set_title(f'Original\n(Digit {y[sample_idx]})', fontsize=9)
axes[0].axis('off')

for ax, n in zip(axes[1:], n_comp_values):
    pca_n = PCA(n_components=n)
    X_compressed = pca_n.fit_transform(X_scaled)
    X_reconstructed = pca_n.inverse_transform(X_compressed)
    X_reconstructed_orig = scaler.inverse_transform(X_reconstructed)
    
    img = X_reconstructed_orig[sample_idx].reshape(8, 8)
    error = np.mean((X[sample_idx] - X_reconstructed_orig[sample_idx])**2)
    
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f'{n} PCs\nMSE={error:.1f}', fontsize=9)
    ax.axis('off')

plt.suptitle('Reconstruction Quality vs Number of PCs', fontsize=11)
plt.tight_layout()
plt.show()

## Section 5: PCA + Classification Pipeline

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Compare different number of PCs
n_components_list = [2, 5, 10, 20, 30, 40, 64]
scores = []

for n in n_components_list:
    if n == 64:
        pipe = Pipeline([('sc', StandardScaler()), ('clf', SVC(gamma='scale'))])
    else:
        pipe = Pipeline([
            ('sc', StandardScaler()),
            ('pca', PCA(n_components=n)),
            ('clf', SVC(gamma='scale'))
        ])
    cv_scores = cross_val_score(pipe, X, y, cv=5)
    scores.append(cv_scores.mean())
    print(f"n_components={n:3d}: accuracy = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

plt.figure(figsize=(9, 4))
plt.plot(n_components_list, scores, 'bo-', linewidth=2, markersize=8)
plt.axhline(scores[-1], color='red', linestyle='--', label=f'No PCA: {scores[-1]:.3f}')
plt.xlabel('Number of PCA Components')
plt.ylabel('Cross-Val Accuracy')
plt.title('SVM Accuracy vs Number of PCA Components')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_n = n_components_list[np.argmax(scores[:-1])]
print(f"\nBest accuracy with {best_n} components: {max(scores[:-1]):.3f}")
print(f"No PCA accuracy (all 64 features): {scores[-1]:.3f}")
print(f"Compression: 64 → {best_n} features ({best_n/64*100:.0f}% of original)")

## Section 6: t-SNE Visualization

In [ ]:
from sklearn.manifold import TSNE

# PCA 2D
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(StandardScaler().fit_transform(X))

# t-SNE (use all 1797 samples)
tsne = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000)
X_tsne = tsne.fit_transform(X)  # t-SNE works on raw data too

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cmap = plt.cm.get_cmap('tab10', 10)
for digit in range(10):
    mask = y == digit
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                    c=[cmap(digit)], label=str(digit), alpha=0.5, s=20)
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=[cmap(digit)], label=str(digit), alpha=0.5, s=20)

axes[0].set_title(f'PCA 2D\n({pca_2d.explained_variance_ratio_.sum()*100:.1f}% variance explained)')
axes[0].legend(title='Digit', bbox_to_anchor=(1, 1), markerscale=2)

axes[1].set_title('t-SNE 2D\n(Clusters clearly separated!)')
axes[1].legend(title='Digit', bbox_to_anchor=(1, 1), markerscale=2)

plt.suptitle('Digits Dataset: PCA vs t-SNE Visualization', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observations:")
print("- PCA 2D: Many digits overlap, hard to separate")
print("- t-SNE: Each digit forms its own island!")
print("- Some digits (3,5,8) may still overlap — they look similar")

In [ ]:
# Adding digit labels to t-SNE plot
fig, ax = plt.subplots(figsize=(10, 8))

for digit in range(10):
    mask = y == digit
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=[cmap(digit)], alpha=0.4, s=20)
    # Add label at cluster center
    center_x = X_tsne[mask, 0].mean()
    center_y = X_tsne[mask, 1].mean()
    ax.annotate(str(digit), (center_x, center_y),
                fontsize=20, fontweight='bold', ha='center',
                color=cmap(digit),
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

ax.set_title('t-SNE — Digits Clusters with Labels', fontsize=13)
ax.set_xlabel('t-SNE Component 1')
ax.set_ylabel('t-SNE Component 2')
plt.tight_layout()
plt.show()

## Section 7: PCA vs t-SNE Final Comparison

In [ ]:
print("=" * 55)
print(f"{'Feature':<25} {'PCA':>14} {'t-SNE':>14}")
print("=" * 55)
rows = [
    ('Type', 'Linear', 'Non-linear'),
    ('Speed (1797 samples)', 'Very fast', 'Moderate'),
    ('Interpretable', 'Yes (loadings)', 'No'),
    ('For ML Pipeline', 'Yes', 'No'),
    ('Visualization', 'Ok', 'Excellent'),
    ('Reproducible', 'Yes', 'Need seed'),
    ('Cluster preservation', 'Partial', 'Strong'),
    ('Global structure', 'Preserves', 'May distort'),
]
for feat, pca_val, tsne_val in rows:
    print(f"{feat:<25} {pca_val:>14} {tsne_val:>14}")
print("=" * 55)

print("\nRule of Thumb:")
print("  Use PCA → when preprocessing for ML or need interpretability")
print("  Use t-SNE → when visualizing clusters in high-D data")

## Section 8: Aaj Humne Kya Seekha?

### Summary:
1. **Digits Dataset** — 64 pixel features pe PCA applied
2. **Eigendigits** — Principal Components ko visualize kiya
3. **Reconstruction** — Compressed → Original image recover
4. **10-20 PCs** — 64 features se similar accuracy
5. **t-SNE** — Much better cluster visualization than PCA
6. **PCA for speed** — Training fast when features are many

### Real-World Uses:
- **Face Recognition (Eigenfaces):** Microsoft, Apple FaceID
- **Image Compression:** JPEG uses PCA-like transform (DCT)
- **Drug Discovery:** 1000s of molecular features → PCA → clustering
- **Finance:** Portfolio analysis — correlated stocks → factors
- **Recommendation Systems:** User-item matrix reduction

---
## Capstone Challenge

**Problem:** Wine dataset pe PCA + classification.

**Tasks:**
1. Load `load_wine()` — 13 features, 178 samples, 3 classes
2. Scale and apply PCA. How many PCs for 95% variance?
3. Plot PCA 2D visualization, colored by wine class
4. Compare accuracy of:
   - Logistic Regression without PCA (13 features)
   - Logistic Regression with PCA (95% variance components)
   - Logistic Regression with PCA (2 components only)
5. Apply t-SNE with 2 components and visualize
6. Which features contribute most to PC1 and PC2?

In [ ]:
# CAPSTONE — TRY YOURSELF FIRST!
from sklearn.datasets import load_wine

wine = load_wine()
X_wine, y_wine = wine.data, wine.target
print(f"Wine: {X_wine.shape}, classes: {wine.target_names}")
print(f"Features: {wine.feature_names}")
print("Your turn! Complete all 6 tasks...")
# YOUR CODE HERE


---
## Solutions — Capstone

In [ ]:
# === SOLUTION ===
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

# Task 2: PCA
scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)

pca_all = PCA()
pca_all.fit(X_wine_scaled)
cumvar = np.cumsum(pca_all.explained_variance_ratio_) * 100
n_95 = np.argmax(cumvar >= 95) + 1
print(f"Task 2: {n_95} components needed for 95% variance")

# Task 3: PCA 2D
pca_2d = PCA(n_components=2)
X_wine_2d = pca_2d.fit_transform(X_wine_scaled)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors_wine = ['steelblue', 'coral', 'green']
for i in range(3):
    mask = y_wine == i
    axes[0].scatter(X_wine_2d[mask, 0], X_wine_2d[mask, 1],
                    c=colors_wine[i], label=wine.target_names[i], alpha=0.7, s=50)
axes[0].set_title(f'PCA 2D\n({pca_2d.explained_variance_ratio_.sum()*100:.1f}% variance)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

# Task 4: Accuracy comparison
configs = [
    ('No PCA (13)', Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])),
    (f'PCA {n_95} comp', Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=n_95)), ('clf', LogisticRegression(max_iter=1000))])),
    ('PCA 2 comp', Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=2)), ('clf', LogisticRegression(max_iter=1000))])),
]

names, accs = [], []
for name, pipe in configs:
    acc = cross_val_score(pipe, X_wine, y_wine, cv=5).mean()
    names.append(name)
    accs.append(acc)
    print(f"Task 4: {name:<20} → {acc:.3f}")

axes[1].bar(names, accs, color=['steelblue', 'coral', 'green'], alpha=0.8)
for i, acc in enumerate(accs):
    axes[1].text(i, acc + 0.005, f'{acc:.3f}', ha='center', fontweight='bold')
axes[1].set_ylim(0.8, 1.05)
axes[1].set_title('Accuracy Comparison')
axes[1].set_ylabel('Cross-Val Accuracy')
axes[1].tick_params(axis='x', rotation=15)

# Task 5: t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=20)
X_wine_tsne = tsne.fit_transform(X_wine_scaled)

for i in range(3):
    mask = y_wine == i
    axes[2].scatter(X_wine_tsne[mask, 0], X_wine_tsne[mask, 1],
                    c=colors_wine[i], label=wine.target_names[i], alpha=0.7, s=50)
axes[2].set_title('t-SNE 2D')
axes[2].legend()

plt.suptitle('Wine Dataset: PCA Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Task 6: Feature loadings
print("\nTask 6: Feature contributions to PC1 and PC2")
pca_n95 = PCA(n_components=n_95)
pca_n95.fit(X_wine_scaled)

pc1_loadings = sorted(zip(wine.feature_names, pca_n95.components_[0]), key=lambda x: abs(x[1]), reverse=True)
print("\nTop features in PC1 (sorted by absolute loading):")
for feat, loading in pc1_loadings[:5]:
    bar = '█' * int(abs(loading) * 20)
    print(f"  {feat:<25} {loading:+.3f} {bar}")